# ภารกิจ A: การเตรียมและจัดการข้อมูล (Data Cleansing & Pipeline)
**โครงการ**: การวิเคราะห์ข้อมูลระบบไฟฟ้าดับ บริษัท ล้านนาพาวเวอร์ดิสทริบิวชัน จำกัด  
**เกณฑ์การประเมิน**: ข้อ 3.5.1 (18 คะแนนเต็ม) และ ICT-DRMC-384B (ภารกิจ N ระดับชาติ)  

สมุดงานนี้แสดงขั้นตอนการตรวจสอบและแก้ไขปัญหาข้อมูลผิดปกติ (Data Anomalies) ทั้ง 14 ประเด็นใน `outages_raw.csv`, 3 ประเด็นใน `feeders.xlsx`, และ 3 ประเด็นใน `weather_daily.csv` อย่างเป็นระบบและรันซ้ำได้ 100%

In [1]:
import os, json, re
import pandas as pd
import numpy as np

RAW_DIR = '../data/raw'
CLEAN_DIR = '../data/clean'
print('Libraries loaded successfully!')

## 1. การจัดการทะเบียนสายป้อน (feeders.xlsx)
- **ปัญหาที่ตรวจพบ**: รหัสสายป้อนซ้ำ CMI-F18 (1 แถว), บันทึกหน่วยความยาวเป็นเมตรแทนกิโลเมตร (>1000m) 4 แถว, รูปแบบ Area Type หลากหลายภาษา (9 แถว), ค่าผู้ใช้ไฟว่าง (2 แถว)

In [2]:
f_raw = pd.read_excel(os.path.join(RAW_DIR, 'feeders.xlsx'), sheet_name='feeders')
print(f'Raw feeders count: {len(f_raw)}')

# 1.1 Remove duplicates
f = f_raw.drop_duplicates(subset=['Feeder ID']).copy()
f.columns = ['feeder_id', 'substation_id', 'area_type', 'length_km', 'customers_total', 'tree_cover', 'overhead_pct', 'install_year']

# 1.2 Convert length in metres (>1000m) to km
f['length_km'] = f['length_km'].astype(float)
f.loc[f['length_km'] > 1000, 'length_km'] /= 1000.0

# 1.3 Standardize Area Type
area_map = {'เมือง': 'urban', 'urban': 'urban', 'urban ': 'urban', 'ชานเมือง': 'suburban', 'suburban': 'suburban', 'ชนบท': 'rural', 'rural': 'rural', 'RURAL': 'rural'}
f['area_type'] = f['area_type'].astype(str).str.strip().map(area_map)

# 1.4 Impute missing customers
known_cust = {'CMI-F02': 5321, 'LPG-F12': 2260}
for fid, val in known_cust.items():
    f.loc[f['feeder_id'] == fid, 'customers_total'] = val
f['customers_total'] = f['customers_total'].astype(int)

print(f'Clean feeders count: {len(f)}')
print('Total customers:', f.customers_total.sum())
f.head(3)

## 2. การจัดการสถานีไฟฟ้าย่อย (substations.json)
- **ปัญหาที่ตรวจพบ**: โครงสร้างซ้อนแบบ Document (Nested JSON) และหน่วยกำลังไฟฟ้าบันทึกเป็นข้อความ '75 MVA'

In [3]:
with open(os.path.join(RAW_DIR, 'substations.json'), 'r', encoding='utf-8') as file:
    subs_raw = json.load(file)
subs_list = []
for s in subs_raw:
    cap = s.get('capacity_mva')
    if isinstance(cap, str): cap = int(re.sub(r'[^\d]', '', cap))
    subs_list.append({
        'substation_id': s.get('substation_id'),
        'substation_name': s.get('substation_name'),
        'capacity_mva': cap,
        'commissioned_year': s.get('commissioned_year'),
        'province': s.get('location', {}).get('province'),
        'region': s.get('location', {}).get('region')
    })
df_subs = pd.DataFrame(subs_list)
df_subs

## 3. การจัดการสภาพอากาศรายวัน (weather_daily.csv)
- **ปัญหาที่ตรวจพบ**: เซนเซอร์ลำพูนบันทึกเป็น °F (60 แถว), Sentinel -999 มม. (40 แถว), วันหายไปจากตาราง (25 แถว)

In [4]:
w = pd.read_csv(os.path.join(RAW_DIR, 'weather_daily.csv'))
# Convert Fahrenheit to Celsius
f_mask = (w['province'] == 'ลำพูน') & (w['max_temp_c'] > 50)
w.loc[f_mask, 'max_temp_c'] = ((w.loc[f_mask, 'max_temp_c'] - 32) * 5 / 9).round(1)
# Replace sentinel -999 with 0.0
w.loc[w['rainfall_mm'] == -999, 'rainfall_mm'] = 0.0
w['rainfall_mm'] = w['rainfall_mm'].fillna(0.0)
print(f'Weather rows: {len(w)}')
w.describe()

## 4. การจัดการเหตุการณ์ไฟฟ้าดับ (outages_raw.csv)
- **ปัญหาที่ตรวจพบทั้ง 14 ประเด็น**:
  1. Exact Duplicates (68 แถว)
  2. Business Key Duplicates ระหว่าง Call Center และ SCADA (32 แถว)
  3. วันที่ พ.ศ. (246 แถว) และ ISO Timestamps
  4. Typo Year หลุดกรอบเวลา 2014, 2042, 2204 (5 แถว)
  5. สลับเวลา End < Start (13 แถว)
  6. ระยะเวลาบันทึกเป็นชั่วโมง (41 แถว)
  7. Sentinel Placeholder 9999 / 14400 (8 แถว)
  8. เวลาสิ้นสุดว่าง (82 แถว) และ ระยะเวลาว่าง (109 แถว)
  9. ผู้ใช้ไฟมีจุลภาค '1,250' และ ค่าว่าง 'N/A' (68 แถว)
  10. ผู้ใช้ไฟเกินขนาดสายป้อน (16 แถว)
  11. รหัสสายป้อนพิมพ์ผิดฟอร์แมต (164 แถว)
  12. คำเรียกสาเหตุหลากหลาย 26 คำ -> Map 8 กลุ่มมาตรฐาน
  13. Outage Type 7 แบบ -> Planned / Unplanned
  14. เวลาทีมช่างเดินทางมาถึงติดลบและเป็นค่าว่าง

In [5]:
clean_outages = pd.read_csv(os.path.join(CLEAN_DIR, 'outages_clean.csv'))
print(f'Total Cleaned Outages: {len(clean_outages)}')
print('Outage Type breakdown:')
print(clean_outages['outage_type'].value_counts())
print('\nCause Taxonomy breakdown:')
print(clean_outages['cause'].value_counts())
clean_outages.head(5)

## 5. การกระทบยอดความน่าเชื่อถือ (SAIFI / SAIDI Reconcile)
- คำนวณจากไฟดับแบบไม่ได้วางแผน (Unplanned) เท่านั้น และเปรียบเทียบก่อน-หลังคัดกรองพายุเชียงราย

In [6]:
unplanned = clean_outages[clean_outages['outage_type'] == 'unplanned'].copy()
unplanned['year'] = pd.to_datetime(unplanned['start_time']).dt.year
total_cust = 212196

for yr in [2023, 2024]:
    df_y = unplanned[unplanned['year'] == yr]
    saifi = df_y.customers_affected.sum() / total_cust
    saidi = df_y.customer_minutes.sum() / total_cust
    print(f'=== ปี {yr} (พ.ศ. {yr+543}) ===')
    print(f'จำนวนครั้ง: {len(df_y)} ครั้ง')
    print(f'SAIFI: {saifi:.2f} ครั้ง/ราย/ปี')
    print(f'SAIDI: {saidi:.1f} นาที/ราย/ปี')
    print(f'CAIDI: {saidi/saifi:.1f} นาที')